# E08 - Continual and Lifelong Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

**Build systems that learn continuously without forgetting**

**MAI Alignment:** COMPSCI 713 (Continual Learning Lecture) | **Next Level:** [E09 - Self-Supervised and Contrastive Learning](./E09%20-%20Self-Supervised%20and%20Contrastive%20Learning.ipynb)

---

## Learning Objectives

1. Explain the principles and challenges of continual learning (CL)
2. Identify and explain the characteristics of Replay-based, Regularisation-based and Architecture-based CL techniques
3. Give examples of Replay-based, Regularisation-based and Architecture-based CL techniques
4. Implement Elastic Weight Consolidation (EWC)
5. Build a replay-based continual learning system
6. Evaluate continual learning performance

**Duration:** 5-6 hours  
**Prerequisites:** E01-E07

---

---

## Learning Progress Tracker

### Completion Status
- [ ] Lesson completed
- [ ] All code cells executed successfully
- [ ] Understood all key concepts
- [ ] Completed practice exercises

### Key Concepts to Remember (E08)
- [ ] Catastrophic forgetting & stability-plasticity dilemma
- [ ] CL settings (class-incremental, domain-incremental, task-incremental)
- [ ] Replay-based methods (iCaRL, GEM)
- [ ] Regularisation-based methods (LwF, EWC)
- [ ] Architecture-based methods (PackNet, PNN)

---

## 1. Overview & Definitions

### What is Continual Learning?

**Continual Learning** (a.k.a. *Lifelong Learning*, *Incremental Learning*) is the ability of a model to learn from a stream of data with **changing data distribution**, with the goal of **transferring knowledge between tasks**.

**Core Idea:** An incremental learner learns continuously from a sequence of tasks, without accessing all data at once, while aiming to:
- **Acquire new knowledge**
- **Retain previously learned knowledge** (avoid forgetting)

> **Reference:** Tercan, H., Deibert, P., & Meisen, T. (2022). Continual learning of neural networks for quality prediction in production using memory aware synapses and weight transfer. *Journal of Intelligent Manufacturing*, 33(1), 283-292.

### Knowledge Transfer in CL

- **Forward transfer:** Knowledge learnt during one task affects learning of *future* tasks
  - Positive forward transfer = learning task A helps learn task B faster
- **Backward transfer:** Knowledge learnt during one task affects performance on *previous* tasks
  - **Catastrophic Forgetting** is *negative* backward transfer
  - Positive backward transfer = learning task B improves performance on task A

### The Stability-Plasticity Dilemma

CL is fundamentally about balancing two competing objectives:

| Property | Definition | Risk if too much |
|----------|-----------|------------------|
| **Plasticity** | Ability to adapt and integrate new information | Rapidly forgets previous tasks |
| **Stability** | Ability to preserve existing representations | Fails to learn new tasks |

---

## 2. Catastrophic Forgetting

### The Formal Problem

We aim to minimise the expected loss over **all tasks seen so far**:

$$\sum_{t=1}^{\mathcal{T}} \mathbb{E}_{(x^t, y^t) \sim \mathcal{D}^t} [\mathcal{L}(f(x^t; \theta), y^t)]$$

In practice, we only have access to data from the **current task** $\mathcal{T}$:

$$\frac{1}{N_{\mathcal{T}}} \sum_{i=1}^{N_{\mathcal{T}}} \ell(f(x_i^{\mathcal{T}}; \theta), y_i^{\mathcal{T}})$$

**When optimising for the current task, we will deteriorate performance on old tasks!**

### Where Can Shift Happen?

Data arrives in multiple stages (tasks). Each task may introduce new:

- **Class shift:** The set of labels or their frequencies change over time  
  *E.g., image classifier trained on cats/dogs later encounters rabbits*

- **Domain shift:** The input representation changes, even if task and labels remain the same  
  *E.g., model trained on sunny photos used on night-time or indoor photos*

- **Task shift:** The objective or prediction target changes, even if domain remains unchanged  
  *E.g., model trained to classify dog/cat later required to count dogs*

---

## 3. Settings of Continual Learning

### Class-Incremental Learning (CIL)
- Model is presented with **new classes** over time
- Must eventually distinguish between **all classes seen so far**
- *E.g., medical image classification where new pathologies are discovered*

### Domain-Incremental Learning (DIL)
- Task remains unchanged, but the **input domain changes** over time
- Model must be robust to these changes **without explicit cues** (Task-ID not available at test time)
- *E.g., classify objects in new environments like car detection across daylight, rain, or snowstorm conditions*

### Task-Incremental Learning (TIL)
- Model exposed to a **sequence of distinct tasks**, learned one at a time
- Task identity is known at test time (multi-head architecture)
- *E.g., robotics where a robot must learn a new function*

| Setting | New Classes? | Domain Changes? | Task-ID at Test? | Difficulty |
|---------|:---:|:---:|:---:|:---:|
| Class-Incremental | Yes | No | No | Hardest |
| Domain-Incremental | No | Yes | No | Medium |
| Task-Incremental | Varies | Varies | Yes | Easiest |

---

## 4. Techniques for Deep Continual Learning

The main approaches to continual learning can be categorised into three families:

```
                    Continual Learning Methods
                    /           |           \
            Replay          Regularisation    Parameter Isolation
           Methods            Methods            Methods
          /      \           /       \          /        \
    Rehearsal  Pseudo    Constrained  Prior   Fixed     Dynamic
              Rehearsal              Focused  Network   Architectures
    iCaRL      DGR       GEM         EWC     PackNet    PNN
    ER         PR        A-GEM       IMM     PathNet    Expert Gate
    SER        CCLUGM    GSS         SI      Piggyback  RCL
    TEM        LGM                   R-EWC   HAT        DAN
    CoPE                             MAS
                                     LwF
                                     LFL
                                     EBLL
                                     DMC
```

> **Reference:** De Lange, M., Aljundi, R., Masana, M., et al. (2021). A continual learning survey: Defying forgetting in classification tasks. *IEEE TPAMI*, 44(7), 3366-3385.

---

## 5. Replay-Based Methods

### The Idea
Replay samples of old tasks when learning new tasks, to defy forgetting. Also applicable to class- and domain-incremental settings.

A memory buffer stores representative samples from previous tasks. When training on a new task, these stored samples are replayed alongside new data.

### 5.1 iCaRL: Incremental Classifier and Representation Learning

**Setting:** Class-Incremental Learning (Rehearsal method)

**Key Ideas:**
- Keeps **exemplar** samples from each class within a **bounded memory**
- For each new batch of classes, the feature extractor/encoder is updated with new class data + stored exemplars
- Does **not** use typical softmax classification (would be biased toward new classes due to imbalanced data)
- Instead uses **nearest class mean classifier** with averages of exemplar features for each class

> **Reference:** Rebuffi, S. A., Kolesnikov, A., Sperl, G., & Lampert, C. H. (2017). iCaRL: Incremental classifier and representation learning. *CVPR*, pp. 2001-2010.

### 5.2 Gradient Episodic Memory (GEM)

**Setting:** Task-Incremental Learning

**Key Ideas:**
- Re-training on stored instances would lead to overfitting
- Instead, GEM treats past task data as defining **constraints** that new parameter updates must satisfy
- Intervenes at the **gradient level**, controlling how optimisation proceeds

**Episodic Memory:**
- A fixed-size memory buffer with budget $M$
- If total number of tasks $T$ is known, $M$ is split evenly: $m = M/T$ samples per task
- Stored samples serve as reference points for preserving past performance

**Constrained Optimisation:**

$$\min_{\theta} \ell(f_\theta(x, t), y) \quad \text{subject to} \quad \ell(f_\theta, M_k) \leq \ell(f_\theta^{t-1}, M_k) \quad \forall k < t$$

Where $M_k$ is the episodic memory for task $k$, and $f_\theta^{t-1}$ is the model after task $t-1$.

**Results:**
- GEM prevents negative backward transfer (forgetting)
- Exhibits slight positive backward transfer (learning new tasks can improve past tasks)
- Shows some positive forward transfer

> **Reference:** Lopez-Paz, D., & Ranzato, M. A. (2017). Gradient episodic memory for continual learning. *NeurIPS*, 30.

### 5.3 Pseudo-Rehearsal & Generative Replay

**The Idea:** Instead of storing old instances, input random data into the network to preserve behaviour:
1. Feed random inputs into the previous model
2. Record its outputs
3. Train the new model to match those outputs

This can be seen as **knowledge distillation** and acts as a functional regulariser.

**Issues:** Curse of dimensionality in large feature spaces; random data may lie far from true distribution.

**Improvement - Generative Replay:** Use generative models (GANs for vision, Transformers for NLP) to generate realistic old data instead of random inputs. Much better preservation of past tasks, but substantial training cost if generative model must be trained from scratch.

---

## 6. Regularisation-Based Methods

### The Idea
Preserve performance on past tasks by **restricting updates** to parameters that are important for previous learning. Use a regularisation term in the loss function to consolidate previous learning: **Loss + penalty term**.

**Advantage over replay-based:** Does not need to store previous instances (privacy-preserving).

### 6.1 Learning without Forgetting (LwF)

**The Idea:** Preserves knowledge from previous tasks using **knowledge distillation**.

**Knowledge Distillation** aids the learning of a model using representations learnt by another model:
- Large teacher model learns to classify images
- Small student model learns from teacher's logits
- Logits contain richer information than a label (e.g., class similarity)

**The Process (LwF applies distillation across tasks):**
1. Before learning a new task, pass new task data through the model trained on previous tasks
2. Record the resulting output logits (soft targets)
3. During training on the new task, augment the loss with a **distillation loss** that penalises deviations from previous task outputs

$$\theta_s^*, \theta_o^*, \theta_n^* \leftarrow \arg\min_{\hat{\theta}_s, \hat{\theta}_o, \hat{\theta}_n} \left( \lambda_o \mathcal{L}_{old}(Y_o, \hat{Y}_o) + \mathcal{L}_{new}(Y_n, \hat{Y}_n) + \mathcal{R}(\hat{\theta}_s, \hat{\theta}_o, \hat{\theta}_n) \right)$$

**When does LwF fail?**
- LwF assumes new task inputs are informative to preserve previous tasks
- **Breaks down under domain shift!** If data distribution changes significantly between tasks, outputs on new task inputs are not informative about earlier tasks

> **Reference:** Li, Z., & Hoiem, D. (2017). Learning without forgetting. *IEEE TPAMI*, 40(12), 2935-2947.

### 6.2 Weight Prior Regularisation

Another approach: regularise network parameters using a prior based on previously learned weights.

**Naive approach - L2 Regularisation:**

$$\mathcal{L}(\theta) = \mathcal{L}_B(\theta) + \sum_i (\theta_i - \theta_{A,i}^*)^2$$

**Problem:** Treats all parameters as equally important!

---

### 6.3 Elastic Weight Consolidation (EWC)

**The Idea:** Find parameters important to the performance of previous tasks, and **stop these from being changed much**.

**How to identify important weights?** Using **parameter gradients** (Fisher Information):
- After training a task, pass data through to identify average gradient magnitude
- Large $F_i$ → important parameter (larger penalty, harder to change)
- Small $F_i$ → flexible parameter (smaller penalty, easier to change)

**EWC Loss Function:**

$$\mathcal{L}(\theta) = \mathcal{L}_B(\theta) + \sum_i \frac{\lambda}{2} F_i (\theta_i - \theta_{A,i}^*)^2$$

Where:
- $\mathcal{L}_B(\theta)$ is the loss for the current task (Task B)
- $F_i$ is the Fisher information for parameter $i$
- $\theta_{A,i}^*$ is the value learned for Task A
- $\lambda$ controls the strength of the regularisation

**Computing $F_i$:**
1. Train on Task A (learn parameters $\theta_A^*$)
2. Use the trained model, pass Task A data through
3. Compute: $F_i = \mathbb{E}\left[\left(\frac{\partial \log p(y|x, \theta)}{\partial \theta_i}\right)^2\right]\bigg|_{\theta=\theta_A^*}$

**Intuition (in parameter space):**
- Blue arrow (no penalty): minimises Task B loss but increases Task A error
- Green arrow (L2 penalty): uniform penalty leads to higher errors on both tasks
- Red arrow (EWC): finds a step lowering Task B error while keeping low error on Task A

**Issues with EWC:**
- **Scalability:** Memory and computation grow with number of tasks
- **Soft constraints:** Penalty may be insufficient for long task sequences
- **Cumulative drift:** Small changes across many parameters accumulate
- **Over-parameterisation sensitivity:** In large networks, identifying truly critical weights is challenging

> **Reference:** Kirkpatrick, J., et al. (2017). Overcoming catastrophic forgetting in neural networks. *PNAS*, 114(13), 3521-3526.

---

## 7. Architecture-Based Approaches

### The Idea
- Use **different sub-networks** for each task
- Typically dedicates different model parameters to each task
- Prevent future tasks from changing previous task weights (**parameter freezing**)
- This can **prevent any possible forgetting!**

### Two Broad Categories

**Static architectures:**
- A fixed-size network is shared across tasks
- Internal mechanisms (e.g., masking) control which parameters are active
- Example: **PackNet**

**Dynamic architectures:**
- The network grows over time by adding new parameters, branches, or modules for each task
- Example: **Progressive Neural Networks (PNN)**

### 7.1 PackNet

**The Idea:** Identify unused or unimportant weights within a fixed network and reallocate them to new tasks.

**Process:** For each task, prune the network weights → re-train on the new task → freeze those task-specific weights.

**Problem 1 - Finite Model Capacity:**
- Will run out of available parameters (inherent to static architectures)
- Alternative: dynamic architecture (but introduces unbounded model growth)

**Problem 2 - Memory Constraints:**
- Requires storing a task-specific mask for each parameter
- Solution: store a binary encoded number for each parameter indicating which task it belongs to
- Only $\log_2(\text{num\_tasks})$ bits per model parameter
- Empirically ~6% increase in memory for VGG-16

> **Reference:** Mallya, A., & Lazebnik, S. (2018). PackNet: Adding multiple tasks to a single network by iterative pruning. *CVPR*, pp. 7765-7773.

---

## 8. Practical Implementation

Now let's implement key continual learning techniques. We'll demonstrate catastrophic forgetting and then show how EWC and replay-based methods mitigate it.

---

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy
import warnings
warnings.filterwarnings("ignore")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

### 8.1 Setting Up Split-MNIST Tasks

We'll split MNIST into 5 binary classification tasks to simulate a class-incremental learning scenario:
- Task 1: Classify digits 0 vs 1
- Task 2: Classify digits 2 vs 3
- Task 3: Classify digits 4 vs 5
- Task 4: Classify digits 6 vs 7
- Task 5: Classify digits 8 vs 9

In [ ]:
def get_split_mnist(task_id, train=True, batch_size=64):
    """Get a binary classification task from MNIST.
    Task i classifies digits 2*i vs 2*i+1."""
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    
    dataset = datasets.MNIST("./data", train=train, download=True, transform=transform)
    
    # Filter for the two classes in this task
    class_a = 2 * task_id
    class_b = 2 * task_id + 1
    
    indices = [i for i, (_, label) in enumerate(dataset) if label in [class_a, class_b]]
    subset = Subset(dataset, indices)
    
    loader = DataLoader(subset, batch_size=batch_size, shuffle=train)
    return loader, (class_a, class_b)


print("Split-MNIST tasks:")
for t in range(5):
    loader, classes = get_split_mnist(t)
    print(f"  Task {t+1}: Classify {classes[0]} vs {classes[1]} ({len(loader.dataset)} samples)")

### 8.2 Define a Simple MLP Model

In [ ]:
class SimpleMLP(nn.Module):
    """Simple MLP for continual learning experiments."""
    def __init__(self, input_size=784, hidden_size=256, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, num_classes)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = x.view(x.size(0), -1)  # Flatten
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x)
    
    def get_features(self, x):
        """Get intermediate features (for analysis)."""
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        return self.relu(self.fc2(x))


model = SimpleMLP().to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

### 8.3 Training and Evaluation Utilities

In [ ]:
def train_task(model, loader, epochs=5, lr=0.001, task_classes=None):
    """Train model on a single task."""
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    model.train()
    
    for epoch in range(epochs):
        total_loss = 0
        correct = 0
        total = 0
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            pred = output.argmax(dim=1)
            correct += pred.eq(target).sum().item()
            total += target.size(0)
    
    return total_loss / len(loader), correct / total


def evaluate_task(model, loader):
    """Evaluate model on a task."""
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            pred = output.argmax(dim=1)
            correct += pred.eq(target).sum().item()
            total += target.size(0)
    return correct / total


print("Training and evaluation utilities defined.")

### 8.4 Demonstrating Catastrophic Forgetting

Let's train a model sequentially on all 5 tasks and observe how performance on earlier tasks degrades.

In [ ]:
# Naive sequential training (no CL strategy)
naive_model = SimpleMLP().to(device)
num_tasks = 5

# Store accuracy matrix: acc_matrix[i][j] = accuracy on task j after training on task i
acc_matrix_naive = np.zeros((num_tasks, num_tasks))

for task_id in range(num_tasks):
    train_loader, classes = get_split_mnist(task_id, train=True)
    print(f"\nTraining on Task {task_id+1} (digits {classes[0]} vs {classes[1]})...")
    loss, acc = train_task(naive_model, train_loader, epochs=5)
    print(f"  Training accuracy: {acc:.4f}")
    
    # Evaluate on all tasks seen so far
    for eval_task in range(num_tasks):
        eval_loader, _ = get_split_mnist(eval_task, train=False)
        acc_matrix_naive[task_id][eval_task] = evaluate_task(naive_model, eval_loader)
    
    print(f"  Accuracies on all tasks: {[f"{a:.3f}" for a in acc_matrix_naive[task_id]]}")

print(f"\nFinal average accuracy: {acc_matrix_naive[-1].mean():.4f}")
print(f"Average forgetting: {(acc_matrix_naive.max(axis=0) - acc_matrix_naive[-1]).mean():.4f}")

### 8.5 Implementing Elastic Weight Consolidation (EWC)

EWC penalises changes to parameters proportional to their importance (measured by Fisher Information).

In [ ]:
class EWC:
    """Elastic Weight Consolidation implementation."""
    
    def __init__(self, model, lambda_ewc=400):
        self.model = model
        self.lambda_ewc = lambda_ewc
        self.saved_params = {}  # theta_A* for each task
        self.fisher_info = {}   # F_i for each task
    
    def compute_fisher(self, data_loader, task_id):
        """Compute Fisher Information Matrix (diagonal approximation)."""
        self.model.eval()
        fisher = {n: torch.zeros_like(p) for n, p in self.model.named_parameters() if p.requires_grad}
        
        for data, target in data_loader:
            data, target = data.to(device), target.to(device)
            self.model.zero_grad()
            output = self.model(data)
            loss = F.cross_entropy(output, target)
            loss.backward()
            
            for n, p in self.model.named_parameters():
                if p.requires_grad and p.grad is not None:
                    fisher[n] += p.grad.data ** 2
        
        # Average over dataset
        for n in fisher:
            fisher[n] /= len(data_loader)
        
        # Save Fisher and parameters for this task
        self.fisher_info[task_id] = fisher
        self.saved_params[task_id] = {n: p.data.clone() for n, p in self.model.named_parameters()}
    
    def penalty(self):
        """Compute EWC penalty term."""
        loss = 0
        for task_id in self.fisher_info:
            for n, p in self.model.named_parameters():
                if n in self.fisher_info[task_id]:
                    fisher = self.fisher_info[task_id][n]
                    old_param = self.saved_params[task_id][n]
                    loss += (fisher * (p - old_param) ** 2).sum()
        return (self.lambda_ewc / 2) * loss


def train_task_ewc(model, loader, ewc, epochs=5, lr=0.001):
    """Train with EWC regularisation."""
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    model.train()
    
    for epoch in range(epochs):
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target) + ewc.penalty()
            loss.backward()
            optimizer.step()


print("EWC implementation defined.")

### 8.6 Training with EWC

In [ ]:
# EWC training
ewc_model = SimpleMLP().to(device)
ewc = EWC(ewc_model, lambda_ewc=400)

acc_matrix_ewc = np.zeros((num_tasks, num_tasks))

for task_id in range(num_tasks):
    train_loader, classes = get_split_mnist(task_id, train=True)
    print(f"\nTraining on Task {task_id+1} (digits {classes[0]} vs {classes[1]}) with EWC...")
    
    train_task_ewc(ewc_model, train_loader, ewc, epochs=5)
    
    # Compute Fisher Information after training on this task
    ewc.compute_fisher(train_loader, task_id)
    
    # Evaluate on all tasks
    for eval_task in range(num_tasks):
        eval_loader, _ = get_split_mnist(eval_task, train=False)
        acc_matrix_ewc[task_id][eval_task] = evaluate_task(ewc_model, eval_loader)
    
    print(f"  Accuracies: {[f"{a:.3f}" for a in acc_matrix_ewc[task_id]]}")

print(f"\nEWC Final average accuracy: {acc_matrix_ewc[-1].mean():.4f}")
print(f"EWC Average forgetting: {(acc_matrix_ewc.max(axis=0) - acc_matrix_ewc[-1]).mean():.4f}")

### 8.7 Implementing Experience Replay

A simple replay-based approach that stores a fixed number of exemplars from each task and replays them during training on new tasks.

In [ ]:
class ReplayBuffer:
    """Simple experience replay buffer for continual learning."""
    
    def __init__(self, max_size_per_task=200):
        self.max_size_per_task = max_size_per_task
        self.buffer_data = []
        self.buffer_targets = []
    
    def add_task_data(self, data_loader):
        """Store exemplars from a task."""
        all_data = []
        all_targets = []
        for data, target in data_loader:
            all_data.append(data)
            all_targets.append(target)
        
        all_data = torch.cat(all_data, dim=0)
        all_targets = torch.cat(all_targets, dim=0)
        
        # Randomly select exemplars
        indices = torch.randperm(len(all_data))[:self.max_size_per_task]
        self.buffer_data.append(all_data[indices])
        self.buffer_targets.append(all_targets[indices])
    
    def get_replay_batch(self, batch_size=32):
        """Get a random batch from the replay buffer."""
        if not self.buffer_data:
            return None, None
        
        all_data = torch.cat(self.buffer_data, dim=0)
        all_targets = torch.cat(self.buffer_targets, dim=0)
        
        indices = torch.randperm(len(all_data))[:batch_size]
        return all_data[indices], all_targets[indices]


def train_task_replay(model, loader, replay_buffer, epochs=5, lr=0.001):
    """Train with experience replay."""
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    model.train()
    
    for epoch in range(epochs):
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            
            # Get replay data
            replay_data, replay_targets = replay_buffer.get_replay_batch(batch_size=32)
            
            if replay_data is not None:
                replay_data = replay_data.to(device)
                replay_targets = replay_targets.to(device)
                # Combine current and replay data
                combined_data = torch.cat([data, replay_data], dim=0)
                combined_targets = torch.cat([target, replay_targets], dim=0)
            else:
                combined_data = data
                combined_targets = target
            
            optimizer.zero_grad()
            output = model(combined_data)
            loss = criterion(output, combined_targets)
            loss.backward()
            optimizer.step()


print("Replay buffer implementation defined.")

### 8.8 Training with Experience Replay

In [ ]:
# Replay-based training
replay_model = SimpleMLP().to(device)
replay_buffer = ReplayBuffer(max_size_per_task=200)

acc_matrix_replay = np.zeros((num_tasks, num_tasks))

for task_id in range(num_tasks):
    train_loader, classes = get_split_mnist(task_id, train=True)
    print(f"\nTraining on Task {task_id+1} (digits {classes[0]} vs {classes[1]}) with Replay...")
    
    train_task_replay(replay_model, train_loader, replay_buffer, epochs=5)
    
    # Add current task data to replay buffer
    replay_buffer.add_task_data(train_loader)
    
    # Evaluate on all tasks
    for eval_task in range(num_tasks):
        eval_loader, _ = get_split_mnist(eval_task, train=False)
        acc_matrix_replay[task_id][eval_task] = evaluate_task(replay_model, eval_loader)
    
    print(f"  Accuracies: {[f"{a:.3f}" for a in acc_matrix_replay[task_id]]}")

print(f"\nReplay Final average accuracy: {acc_matrix_replay[-1].mean():.4f}")
print(f"Replay Average forgetting: {(acc_matrix_replay.max(axis=0) - acc_matrix_replay[-1]).mean():.4f}")

### 8.9 Comparing Methods: Visualising Forgetting

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

titles = ["Naive (No CL Strategy)", "EWC", "Experience Replay"]
matrices = [acc_matrix_naive, acc_matrix_ewc, acc_matrix_replay]

for ax, title, matrix in zip(axes, titles, matrices):
    im = ax.imshow(matrix, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
    ax.set_xlabel("Task Evaluated")
    ax.set_ylabel("After Training Task")
    ax.set_title(title)
    ax.set_xticks(range(num_tasks))
    ax.set_yticks(range(num_tasks))
    ax.set_xticklabels([f"T{i+1}" for i in range(num_tasks)])
    ax.set_yticklabels([f"T{i+1}" for i in range(num_tasks)])
    
    # Add text annotations
    for i in range(num_tasks):
        for j in range(num_tasks):
            ax.text(j, i, f"{matrix[i][j]:.2f}", ha="center", va="center", fontsize=8)

plt.colorbar(im, ax=axes, shrink=0.8, label="Accuracy")
plt.suptitle("Accuracy Matrix: Performance on Task j After Training on Task i", fontsize=13)
plt.tight_layout()
plt.show()

# Summary comparison
print("\n" + "="*60)
print(f"{"Method":<20} {"Final Avg Acc":<18} {"Avg Forgetting":<18}")
print("="*60)
print(f"{"Naive":<20} {acc_matrix_naive[-1].mean():<18.4f} {(acc_matrix_naive.max(axis=0) - acc_matrix_naive[-1]).mean():<18.4f}")
print(f"{"EWC":<20} {acc_matrix_ewc[-1].mean():<18.4f} {(acc_matrix_ewc.max(axis=0) - acc_matrix_ewc[-1]).mean():<18.4f}")
print(f"{"Replay":<20} {acc_matrix_replay[-1].mean():<18.4f} {(acc_matrix_replay.max(axis=0) - acc_matrix_replay[-1]).mean():<18.4f}")
print("="*60)

## 9. Summary & Method Comparison

| Method Family | Approach | Advantages | Disadvantages | Examples |
|--------------|----------|-----------|---------------|----------|
| **Replay-based** | Store/generate old task samples | Simple, effective, works across settings | Memory cost, privacy concerns | iCaRL, GEM, A-GEM |
| **Regularisation-based** | Penalise changes to important parameters | No data storage needed (privacy) | Soft constraints may be insufficient, scalability | EWC, LwF, SI, MAS |
| **Architecture-based** | Dedicate parameters per task | Can prevent all forgetting | Finite capacity (static) or unbounded growth (dynamic) | PackNet, PNN, HAT |

### Key Challenges Remaining

- Catastrophic forgetting is still a major challenge, especially for **class-incremental learning**
- **Knowledge transfer** — limited work on what to transfer and how to transfer forward and backward
- **Open world continual learning** is an even bigger challenge (novelty detection + adaptation + incremental learning)

### Goal: AI Autonomy
The next generation of AI will require autonomous learning agents built in restricted environments, interacting with people, other agents, and the environment — all requiring robust continual learning.

---

## 10. Exercises

### Exercise 1: Tune EWC Lambda
Experiment with different values of `lambda_ewc` (e.g., 10, 100, 1000, 5000). How does it affect the stability-plasticity trade-off?

### Exercise 2: Implement Online EWC
Standard EWC stores Fisher information for each task separately (memory grows linearly). Implement **Online EWC** which maintains a running average of Fisher information across tasks.

### Exercise 3: Nearest Class Mean Classifier
Implement the iCaRL-style nearest class mean classifier. Instead of using softmax outputs, classify by finding the nearest class prototype in feature space.

### Exercise 4: GEM Constraint
Implement the GEM gradient projection constraint: if the current gradient would increase loss on past tasks (negative dot product with past task gradients), project it onto the feasible region.

### Exercise 5: Compare on Permuted MNIST
Create a domain-incremental benchmark using Permuted MNIST (each task applies a different fixed permutation to pixel positions). Compare naive, EWC, and replay methods.

---

## References

1. Kirkpatrick, J., et al. (2017). Overcoming catastrophic forgetting in neural networks. *PNAS*, 114(13), 3521-3526.
2. Rebuffi, S. A., et al. (2017). iCaRL: Incremental classifier and representation learning. *CVPR*, pp. 2001-2010.
3. Lopez-Paz, D., & Ranzato, M. A. (2017). Gradient episodic memory for continual learning. *NeurIPS*, 30.
4. Li, Z., & Hoiem, D. (2017). Learning without forgetting. *IEEE TPAMI*, 40(12), 2935-2947.
5. Mallya, A., & Lazebnik, S. (2018). PackNet: Adding multiple tasks to a single network by iterative pruning. *CVPR*, pp. 7765-7773.
6. De Lange, M., et al. (2021). A continual learning survey: Defying forgetting in classification tasks. *IEEE TPAMI*, 44(7), 3366-3385.
7. Tercan, H., Deibert, P., & Meisen, T. (2022). Continual learning of neural networks for quality prediction in production using memory aware synapses and weight transfer. *Journal of Intelligent Manufacturing*, 33(1), 283-292.

---

*Lecture content adapted from Yun Sing Koh's lecture on Continual Learning (COMPSCI 713, University of Auckland, Instructor: Thomas Lacombe, 22/05/2026)*